In [4]:
!pip install -q "transformers==4.57.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
!pip install -q 'lm_eval[math]' accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you ha

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "tomg-group-umd/huginn-0125"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, torch_dtype=torch.bfloat16, trust_remote_code=True
).cuda().eval()

ids = tok("The capital of Westphalia is the", return_tensors="pt").input_ids.cuda()
with torch.no_grad():
    out = model(ids, num_steps=32)
print(tok.decode(out.logits[0, -1].argmax()))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 city


In [3]:
del model; torch.cuda.empty_cache()

In [2]:
!lm_eval --model hf \
  --model_args pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=bfloat16,mean_recurrence=1 \
  --tasks hellaswag --batch_size=auto --num_fewshot=0 --limit 100

2026-10-08:00:17:28 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-08:00:17:40 INFO     [_cli.run:392] Selected Tasks: ['hellaswag']
2026-10-08:00:17:42 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-08:00:17:42 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'tomg-group-umd/huginn-0125', 'trust_remote_code': True, 'dtype': 'bfloat16', 'mean_recurrence': 1}
2026-10-08:00:17:44 INFO     [models.huggingface:289] Using device 'cuda:0'
`torch_dtype` is deprecated! Use `dtype` instead!
2026-10-08:00:17:45 INFO     [models.huggingface:717] Model type cannot be determined. Using default model type 'causal'
2026-10-08:00:17:45 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
I0000 0

In [ ]:
!lm_eval --model hf \
  --model_args pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=bfloat16,mean_recurrence=32 \
  --tasks hellaswag,arc_challenge --batch_size=auto --num_fewshot=0 \
  --output_path /content/drive/MyDrive/huginn/zeroshot_r32

In [ ]:
!lm_eval --model hf \
  --model_args pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=bfloat16,mean_recurrence=32 \
  --tasks gsm8k_cot --batch_size=auto --apply_chat_template=True --fewshot_as_multiturn \
  --system_instruction="You are a helpful assistant that can assist users with mathematical reasoning." \
  --output_path /content/drive/MyDrive/huginn/gsm8k_cot_r32

2026-10-08:00:50:22 INFO     [_cli.run:392] Selected Tasks: ['gsm8k_cot']
2026-10-08:00:50:23 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-08:00:50:23 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'tomg-group-umd/huginn-0125', 'trust_remote_code': True, 'dtype': 'bfloat16', 'mean_recurrence': 32}
2026-10-08:00:50:27 INFO     [models.huggingface:289] Using device 'cuda:0'
`torch_dtype` is deprecated! Use `dtype` instead!
2026-10-08:00:50:28 INFO     [models.huggingface:717] Model type cannot be determined. Using default model type 'causal'
2026-10-08:00:50:28 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
I0000 00:00:1791420629.929111   41468 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-criti

In [ ]:
!HF_ALLOW_CODE_EVAL=1 lm_eval --model hf \
  --model_args pretrained=tomg-group-umd/huginn-0125,trust_remote_code=True,dtype=bfloat16,mean_recurrence=32 \
  --tasks humaneval_instruct --batch_size=1 --num_fewshot=0 \
  --apply_chat_template=True --confirm_run_unsafe_code \
  --gen_kwargs=do_sample=True,temperature=0.2,top_p=0.95 \
  --output_path /content/drive/MyDrive/huginn/humaneval_r32